In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data-scrap\\data\\query_ساینا_details.csv")

df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def persian_to_english_digits(text: str) -> str:
    if not text:
        return text
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    arabic_digits = "٠١٢٣٤٥٦٧٨٩"
    english_digits = "0123456789"
    trans_table = str.maketrans(
        persian_digits + arabic_digits,
        english_digits + english_digits,
    )
    return text.translate(trans_table)

year_col_idx = df.columns.get_loc("year_text")

df.insert(year_col_idx+1, "model_year", df["year_text"].apply(persian_to_english_digits))

df.insert(
    year_col_idx+2,
    "miladi_year",
    pd.to_numeric(df["model_year"].str.extract(r"-\s*(\d{4})")[0], errors="coerce")
)

df.insert(
    year_col_idx+3,
    "jalali_year",
    pd.to_numeric(df["model_year"].str.extract(r"(\d{4})\s*-")[0], errors="coerce")
)

df

In [ ]:
import datetime
current_year = datetime.date.today().year

havale_idx = df[
                (df["title"].str.contains("حواله|اماده|آماده|تحویل", na=False))
                & (df["mileage_km"] == 0)
                & ( (df["miladi_year"] == current_year) | (df["miladi_year"] == current_year-1) | (df["miladi_year"] == current_year-2) )
            ].index

# df.loc[havale_idx]

# df = df.drop(havale_idx)
df.drop(havale_idx, inplace=True)

df.reset_index(drop=True, inplace=True)

df

In [ ]:
df.info()

In [ ]:
def get_gearbox(row):
    if pd.notna(row["gearbox"]):
        return row["gearbox"]

    model = row["brand_model_text"]

    if pd.isna(model):
        return None

    model = str(model).replace("ي", "ی").replace("ك", "ک")
    model = model.replace("‌", " ")

    if "اتومات" in model:
        return "اتومات"

    if "دنده" in model or "دستی" in model:
        return "دنده‌ای"

    return None

df["gearbox"] = df.apply(get_gearbox, axis=1)

df.info()

In [ ]:
df[df["gearbox"].isna()]

In [ ]:
df.drop(df[df["gearbox"].isna()].index, inplace=True)

df.reset_index(drop=True, inplace=True)

df.info()
# df

In [ ]:
CITY_NAME = ["بوشهر", "تهران",]  # "شیراز", "اصفهان"

col_idx = df.columns.get_loc("location")

df.insert(
    col_idx + 1,
    "city_location_persian",
    df["posted_time_text"].apply(lambda x: next((city for city in CITY_NAME if city in x), None))  # CITY_NAME if CITY_NAME in x else None
)

df

In [ ]:
fdf = df[["token", "brand_model_text", "color", "jalali_year", "miladi_year", "gearbox", "fuel_type", "mileage_km", "engine_condition", "chassis_front_condition", "chassis_rear_condition", "body_condition", "gearbox_condition", "base_price_toman"]].copy()  # "city_location_persian"
fdf

In [ ]:
fdf.columns, fdf.info()

In [ ]:
fdf["base_price_toman"].hist()

In [ ]:
features = [
    "brand_model_text",
    "jalali_year",
    "miladi_year",
    "gearbox",
    "fuel_type",
    "mileage_km",
    "engine_condition",
    "chassis_front_condition",
    "chassis_rear_condition",
    "body_condition",
    "gearbox_condition",
    # "base_price_toman"
]


for feature in features:
    print(f"\n{fdf[feature].value_counts()}")

In [ ]:
import matplotlib.pyplot as plt
from math import ceil

target = "base_price_toman"

cols = 3
rows = ceil(len(features) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4), gridspec_kw={"hspace": 0.6})

for ax, feature in zip(axes.flat, features):
    ax.scatter(fdf[feature], fdf[target], color="blue")
    ax.set_xlabel(feature)
    ax.set_ylabel(target)
    ax.set_title(f"{feature} vs {target}")

plt.tight_layout()
plt.show()

In [ ]:
numeric_features = ["jalali_year", "miladi_year", "mileage_km"]
categorical_features = [
    "brand_model_text", "gearbox", "fuel_type", "engine_condition",
    "chassis_front_condition", "chassis_rear_condition",
    "body_condition", "gearbox_condition",
]

# scatter برای numeric
fig, axes = plt.subplots(1, len(numeric_features), figsize=(15, 4))
for ax, feature in zip(axes, numeric_features):
    ax.scatter(fdf[feature], fdf[target], alpha=0.5)
    ax.set_xlabel(feature)
    ax.set_ylabel(target)

plt.tight_layout()
plt.show()

# boxplot برای categorical
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feature in zip(axes.flat, categorical_features):
    fdf.boxplot(column=target, by=feature, ax=ax, rot=45)
    ax.set_title(feature)

plt.tight_layout()
plt.show()